Instructions:
- Installation : if using Linux or Mac with conda: conda create -n fenicsx-env;
conda activate fenicsx-env;
conda install -c conda-forge fenics-dolfinx mpich pyvista scipy python-gmsh matplotlib scikit-learn;
- Sections : 1, 2 Fenicsx depedent

0) Imports (non - Fenicsx)

In [6]:
import pyvista as pv # Plotting
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
from timeit import default_timer as timer 

1) Imports (Fenicsx)

In [ ]:

from dolfinx import mesh, fem, io, plot
import dolfinx.fem.petsc as petsc
from dolfinx.fem.petsc import assemble_matrix, assemble_vector
import ufl
from mpi4py import MPI

2.1) Auxiliary (not so important functions)

In [7]:
facm = 0.5*np.sqrt(2.)
def eps(u):
    return ufl.sym(ufl.grad(u))

def ten2man(A):
    return ufl.as_vector([A[0,0], A[1,1], facm*(A[0,1] + A[1,0])])

def petsc2scipy(A, shape = None):
    Ai, Aj, Av = A.getValuesCSR()
    A_scipy = sp.csr_matrix((Av, Aj, Ai), shape = shape)
    return A_scipy

def read_mesh(msh_file, gdim = 2):
    # Usage of meshdata
    #domain = meshdata.mesh
    #cell_tags = meshdata.cell_tags
    #facet_tags = meshdata.facet_tags
    #physical = meshdata.physical_groups
    
    meshdata = io.gmsh.read_from_msh(msh_file, MPI.COMM_WORLD, gdim=gdim)
    return meshdata

def pyvista_warp_plot(sol_u, V, gdim = 2, scale_fac = 1.0):
    uh = fem.Function(V)
    uh.x.array[:] = sol_u
    
    topology, cell_types, geometry = plot.vtk_mesh(uh.function_space)
    grid = pv.UnstructuredGrid(topology, cell_types, geometry)

    # Displacement vector field
    u_vec = uh.x.array.reshape((-1, gdim))

    # Magnitude
    u_mag = np.linalg.norm(u_vec, axis=1)
    grid.point_data["|u|"] = u_mag    
    grid.point_data["Displacement"] = np.hstack((u_vec , np.zeros_like(u_vec[:,0]).reshape((-1,1))))

    warped = grid.warp_by_vector("Displacement", factor=scale_fac)

    plotter = pv.Plotter()

    plotter.add_mesh(
        warped,
        scalars="|u|",
        show_edges=True,
        cmap="viridis"
    )
        
    plotter.add_mesh(
        grid,
        style="wireframe",
        color="gray",
        line_width=0.2,
        label="Undeformed"
    )
    

    plotter.view_xy()
    plotter.enable_parallel_projection()
    plotter.show()


**2.2) (Non-DDCM) Variational Problem**
It gets:
$$ a(\bf u, \bf v) = \int_{\Omega} \boldsymbol{\sigma}(\bf u) : \boldsymbol{\varepsilon}(\mathbf{v}) $$

$$ L(\bf v) = \int_{\Gamma_N} \boldsymbol{\sigma}(\bf u) (\mathbf{v}) $$

In [8]:
def get_problem(meshdata, matparam, q_load, CLAMPED_FLAG, LOAD_FLAG, gdim = 2, porder = 1):
    E = matparam['E']
    nu = matparam['nu']
    
    domain = meshdata.mesh
    cell_tags = meshdata.cell_tags
    facet_tags = meshdata.facet_tags
    physical = meshdata.physical_groups
    
    dx = ufl.Measure("dx", domain=domain, subdomain_data=cell_tags)
    ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
    
    # -------------------------
    # Function space (vector)
    # -------------------------
    V = fem.functionspace(domain, ("CG", porder, (gdim,)))
    
    # -------------------------
    # Boundary conditions
    # -------------------------
    # Left: fully fixed
    facets = facet_tags.find(physical[CLAMPED_FLAG].tag)
    dofs = fem.locate_dofs_topological(V, physical[CLAMPED_FLAG].dim, facets)
    u_left = np.array([0.0, 0.0], dtype=np.float64)
    bc_left = fem.dirichletbc(u_left, dofs, V)
    
    bcs = [bc_left]
    
    # -------------------------
    # Material model
    # -------------------------
    mu = E / (2.0 * (1.0 + nu))
    lmbda = E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
    
    lmbda_ = fem.Constant(domain, lmbda)
    mu_ = fem.Constant(domain, mu)
    
    # -------------------------
    # Variational formulation
    # -------------------------
    u = ufl.TrialFunction(V)
    v = ufl.TestFunction(V)
    
    sigma = lmbda_ * ufl.tr(eps(u)) * ufl.Identity(gdim) + 2.0 * mu_ * eps(u)
    a = ufl.inner(sigma, eps(v)) * dx
    
    # External tractions
    t_right = fem.Constant(domain, np.array([q_load, 0.0], dtype=np.float64))
    L = ufl.dot(t_right, v) * ds(physical[LOAD_FLAG].tag)


    return a, L, bcs, V